## Rule 3: Hiding Path Information

Based on the patterns in the BIRD dev set, we will identify tables that serve as bridges and exclude their information from the corresponding questions in the BIRD dev set

## 1. Retrieving the dev questions with their represented subgraphs for each database


In [76]:
import pickle
from pathlib import Path
import numpy as np

def retrieve_all_dev_patterns(folder_path):
    """
    Retrieve all subgraphs from the pickle files in the folder.
    
    Args:
        folder_path (str): The path to the folder containing the .pkl files.
        
    Returns:
        list: A list of all subgraphs from the folder.
    """
    folder = Path(folder_path)
    all_subgraphs = []
    
    for file_name in folder.glob('join_patterns_*.pkl'):
        try:
            with file_name.open('rb') as f:
                join_patterns = pickle.load(f)
                if isinstance(join_patterns, list):
                    all_subgraphs.extend(join_patterns)
        except (ValueError, KeyError, pickle.UnpicklingError) as e:
            print(f"Skipping file {file_name.name}: {e}")
    
    return all_subgraphs

def split_queries(database_subgraphs, train_ratio=0.6, dev_ratio=0.2):
    """
    Splits the equivalent queries into train, dev, and test sets.

    Args:
        database_subgraphs (dict): Dictionary of database subgraphs.
        train_ratio (float): Proportion of data to allocate for training.
        dev_ratio (float): Proportion of data to allocate for development.
    
    Returns:
        tuple: (train_queries, dev_queries, test_queries)
    """
    all_dev_queries = []
    for subgraphs in database_subgraphs.values():
        for subgraph in subgraphs:
            all_dev_queries.extend(subgraph.get('equivalent_queries', []))

    np.random.shuffle(all_dev_queries)
    
    train_size = int(train_ratio * len(all_dev_queries))
    dev_size = int(dev_ratio * len(all_dev_queries))
    
    train_queries = all_dev_queries[:train_size]
    dev_queries = all_dev_queries[train_size:train_size + dev_size]
    test_queries = all_dev_queries[train_size + dev_size:]

    return train_queries, dev_queries, test_queries

def filter_subgraphs(database_subgraphs, queries_list):
    """
    Filters subgraphs to keep only those containing queries in the given set.

    Args:
        database_subgraphs (dict): Dictionary of database subgraphs.
        queries_list (list): List of queries to keep.
    
    Returns:
        dict: Filtered subgraphs.
    """
    filtered_subgraphs = {db_id: [] for db_id in database_subgraphs}
    
    for db_id, subgraphs in database_subgraphs.items():
        for subgraph in subgraphs:
            equivalent_queries = [q for q in subgraph.get('equivalent_queries', []) if q in queries_list]
            if equivalent_queries:
                filtered_subgraphs[db_id].append({
                    'subgraph': subgraph['subgraph'],
                    'pattern_signature': subgraph['pattern_signature'],
                    'equivalent_queries': equivalent_queries
                })
    
    return filtered_subgraphs

# Define database IDs
# db_ids = [
#     "california_schools", "card_games", "codebase_community", "debit_card_specializing",
#     "european_football_2", "financial", "formula_1", "student_club", "superhero",
#     "thrombosis_prediction", "toxicology"
# ]
db_ids = [
    "codebase_community"
]

# Load database subgraphs
database_subgraphs = {
    db_id: retrieve_all_dev_patterns(f'../../../../data/rule_inputs/rules_1_2/{db_id}/')
    for db_id in db_ids
}

# Split queries into train, dev, and test sets
np.random.seed(42)  # Ensuring reproducibility
train_queries, dev_queries, test_queries = split_queries(database_subgraphs)
# Filter subgraphs for each set
train_subgraphs = filter_subgraphs(database_subgraphs, train_queries)
dev_subgraphs = filter_subgraphs(database_subgraphs, dev_queries)
test_subgraphs = filter_subgraphs(database_subgraphs, test_queries)

print(f"Train: {len(train_queries)} queries, {sum(len(subgraphs) for subgraphs in train_subgraphs.values())} subgraphs")
print(f"Dev: {len(dev_queries)} queries, {sum(len(subgraphs) for subgraphs in dev_subgraphs.values())} subgraphs")
print(f"Test: {len(test_queries)} queries, {sum(len(subgraphs) for subgraphs in test_subgraphs.values())} subgraphs")




Train: 111 queries, 22 subgraphs
Dev: 37 queries, 19 subgraphs
Test: 38 queries, 17 subgraphs


## 2. Identifying the candidate table to hide (the most central table to the query)

In [78]:
import pickle
import re
import networkx as nx
from pathlib import Path

def find_central_table_and_components(subgraph, schema):
    """Find the central table in a subgraph and extract its components."""
    if subgraph.number_of_nodes() < 3:
        return "", []

    # Compute betweenness centrality
    betweenness_centrality = nx.betweenness_centrality(subgraph)
    max_betweenness = max(betweenness_centrality.values(), default=0)
    
    # Get candidate tables
    candidates = [node for node, score in betweenness_centrality.items() if score == max_betweenness]
    
    if not candidates:
        return "", []

    # Resolve ties using schema centrality
    if len(candidates) > 1:
        schema_centrality = nx.degree_centrality(schema)
        central_table = max(candidates, key=lambda node: schema_centrality.get(node, 0))
    else:
        central_table = candidates[0]

    # Extract components from table name
    components = re.findall(r'[A-Z]?[a-z]+|[A-Z]+(?=[A-Z][a-z])|\d+[a-zA-Z]*', central_table)
    components = [comp.lower() for comp in components if comp != '_']
    
    return central_table, components

def process_subgraphs(subgraphs, schema):
    """Find central tables, extract components, and filter equivalent queries."""
    kept_subgraphs = []
    
    for subgraph in subgraphs:
        nx_subgraph = subgraph.get("subgraph")
        if not nx_subgraph:
            continue
        
        central_table, components = find_central_table_and_components(nx_subgraph, schema)
        if not central_table:
            continue
        
        # Assign central table and components
        subgraph["central_table"] = central_table
        subgraph["components"] = components
        subgraph['equivalent_queries'] = [
            query for query in subgraph['equivalent_queries']
            if any(comp in query['question'].lower() for comp in components)
        ]
        if subgraph['equivalent_queries']:  # Only keep if queries remain
            kept_subgraphs.append(subgraph)
    
    return kept_subgraphs

def process_dataset(dataset_subgraphs):
    """Process train/dev/test datasets by loading schemas and filtering subgraphs."""
    processed_subgraphs = {}
    
    for db_id, subgraphs in dataset_subgraphs.items():
        schema_path = Path(f'../../../../data/graph_data/bird_graphs/pickles/{db_id}_graph.pkl')
        
        if not schema_path.exists():
            print(f"Warning: Missing schema file for {db_id}")
            continue

        with schema_path.open('rb') as f:
            schema = pickle.load(f)

        processed_subgraphs[db_id] = process_subgraphs(subgraphs, schema)
    
    return processed_subgraphs

# Process all datasets
train_subgraphs_with_central_tables = process_dataset(train_subgraphs)
dev_subgraphs_with_central_tables = process_dataset(dev_subgraphs)
test_subgraphs_with_central_tables = process_dataset(test_subgraphs)




## 3. Hiding the path information using three different techniques
The hiding part is done using GPT-4o. You can use your API key stored in a `.env` file located here.

In [79]:
import openai
import os
from dotenv import load_dotenv

load_dotenv()

openai.api_key = os.getenv("OPENAI_API_KEY")

def apply_synonym_replacement(db_id, question, evidence, sql_query, candidate_table, components):
    """Uses GPT-4o to replace references to candidate tables in questions and evidence using synonyms."""
    
    prompt = f"""
You are in a text-to-SQL context where each question corresponds to a SQL query based on a given database schema.

Your task is to conceal any direct reference to candidate table names in the question and the evidence while keeping them meaningful.  
Use the Synonym Replacement (SR) technique to replace words or phrases linked to the table names with their synonyms.  

**Guidelines:**
- The synonym should not refer to the candidate table itself (e.g., if the table is *cards*, do not use *playing cards*).
- Keep the meaning of the question and evidence intact.
- If the evidence directly mentions a table (e.g., `cards.name`), replace the table name with a synonym and imply the source with pronouns or descriptive phrases.
- If the evidence is empty, return an empty string.
- The **Candidate Table** may be a single word (e.g., *users*) or a compound name (e.g., *postHistory*).  
- You will be given **Components**, a list of parts of the candidate table name. Use them to identify what to replace.
- Do **not** mention the original table name in any way.

**Input:**
- **Database ID:** {db_id}
- **Question:** {question}
- **Evidence:** {evidence}
- **SQL Query:** {sql_query}
- **Candidate Table:** {candidate_table}
- **Components:** {components}

**Output Format:**  
New Question: modified question  
New Evidence: modified evidence (or empty string if none)
"""

    messages = [{"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": prompt}]

    response = openai.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        max_tokens=300,
        temperature=0.7
    )

    generated_text = response.choices[0].message.content.strip()
    
    # Extract new question and evidence
    new_question, new_evidence = "", ""
    for row in generated_text.split("\n"):
        if row.startswith("New Question:"):
            new_question = row.replace("New Question: ", "").strip()
        elif row.startswith("New Evidence:"):
            new_evidence = row.replace("New Evidence: ", "").strip()

    return new_question, new_evidence

def generate_synonym_replacement_queries(db_id, subgraphs):
    """Processes queries by applying synonym replacement and filtering unchanged ones."""
    
    rewritten_queries = []
    print(f"Generating new questions for {db_id}...")

    for subgraph in subgraphs:
        for query in subgraph['equivalent_queries']:
            new_question, new_evidence = apply_synonym_replacement(
                db_id, query['question'], query['evidence'], query['SQL'],
                subgraph['central_table'], subgraph['components']
            )

            # Ensure replacement occurred before keeping it
            if not any(comp in new_question.lower() for comp in subgraph['components']):
                query['new_question'], query['new_evidence'] = new_question, new_evidence
                rewritten_queries.append(query)

    return rewritten_queries

def hide_tables_with_synonym_replacement(data):
    """Runs synonym replacement on the dataset and returns the modified queries."""
    
    return {
        db_id: generate_synonym_replacement_queries(db_id, subgraphs)
        for db_id, subgraphs in data.items()
    }

# ---- Execution Block ----
train_queries_with_synonym_replacement = hide_tables_with_synonym_replacement(train_subgraphs_with_central_tables)
dev_queries_with_synonym_replacement = hide_tables_with_synonym_replacement(dev_subgraphs_with_central_tables)


Generating new questions for codebase_community...
Generating new questions for codebase_community...


In [80]:
import openai
import os
from dotenv import load_dotenv

load_dotenv()

openai.api_key = os.getenv("OPENAI_API_KEY")

def apply_backtranslation(db_id, question, evidence, sql_query, candidate_table, components):
    """Uses GPT-4o to perform back translation with synonym replacement."""
    
    prompt = f"""
You are in a text-to-SQL context where each question corresponds to a SQL query based on a given database schema.

Your task is to conceal any direct reference to the candidate table names in the question and the evidence while keeping it meaningful.  
Use the **Back Translation (BT)** technique. First, translate the question and the evidence into French, then back to English. During the French translation, replace any reference to the candidate table with a relevant synonym.  

**Important Guidelines:**
- Do not use synonyms that refer directly to the candidate table (e.g., if the table is *cards*, do not use *cartes de jeu*).  
- Hide only the table name reference and keep the rest of the question and evidence intact.  
- If the evidence directly mentions a table name (e.g., `cards.name`), replace the table name with a suitable synonym and imply the source with pronouns or descriptive phrases.
- If the evidence is empty, return an empty string.
- The **Candidate Table** may be a single word (e.g., *users*) or a compound name (e.g., *postHistory*).  
- The **Components** list helps identify which words to replace.

**Input:**
- **Database ID:** {db_id}
- **Question:** {question}
- **Evidence:** {evidence}
- **SQL Query:** {sql_query}
- **Candidate Table:** {candidate_table}
- **Components:** {components}

**Output Format:**  
Question in English: [modified question]  
Evidence in English: [modified evidence] (or empty string if none)
"""

    messages = [{"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": prompt}]

    response = openai.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        max_tokens=300,
        temperature=0.7
    )

    generated_text = response.choices[0].message.content.strip()
    
    # Extract new question and evidence
    new_question, new_evidence = "", ""
    for row in generated_text.split("\n"):
        if row.startswith("Question in English:"):
            new_question = row.replace("Question in English: ", "").strip()
        elif row.startswith("Evidence in English:"):
            new_evidence = row.replace("Evidence in English: ", "").strip()

    return new_question, new_evidence

def generate_backtranslated_queries(db_id, subgraphs):
    """Processes queries by applying back translation and filtering unchanged ones."""
    
    rewritten_queries = []
    print(f"Generating backtranslated questions for {db_id}...")

    for subgraph in subgraphs:
        for query in subgraph['equivalent_queries']:
            new_question, new_evidence = apply_backtranslation(
                db_id, query['question'], query['evidence'], query['SQL'],
                subgraph['central_table'], subgraph['components']
            )

            # Ensure replacement occurred before keeping it
            if not any(comp in new_question.lower() for comp in subgraph['components']):
                query['new_question'], query['new_evidence'] = new_question, new_evidence
                rewritten_queries.append(query)

    return rewritten_queries

def hide_tables_with_backtranslation(data):
    """Runs back translation on the dataset, concealing table names."""
    
    return {
        db_id: generate_backtranslated_queries(db_id, subgraphs)
        for db_id, subgraphs in data.items()
    }

# ---- Execution Block ----
train_queries_with_backtranslation = hide_tables_with_backtranslation(train_subgraphs_with_central_tables)
dev_queries_with_backtranslation = hide_tables_with_backtranslation(dev_subgraphs_with_central_tables)



Generating backtranslated questions for codebase_community...
Generating backtranslated questions for codebase_community...


In [81]:
import openai
import os
from dotenv import load_dotenv

load_dotenv()

openai.api_key = os.getenv("OPENAI_API_KEY")

def augment_with_contextual_synonyms(db_id, question, evidence, sql_query, candidate_table, components):
    """Uses GPT-4o to perform contextual augmentation by replacing table references with synonyms."""
    
    prompt = f"""
You are in a text-to-SQL context where each question corresponds to a SQL query based on a given database schema.

Your task is to conceal any direct reference to the candidate table names in the question and evidence while ensuring the sentence remains meaningful.  
Use **Contextual Augmentation (CA)** to replace words or phrases related to the table names with appropriate synonyms that fit naturally within the surrounding context.  

**Guidelines:**  
- Synonyms must be relevant to the database context (e.g., for *card_games*, *cards* could become *playing pieces* but not *vouchers*).  
- Synonyms should not directly reference the candidate table (e.g., *cards* should not become *playing cards*).  
- Ensure the table name is removed while keeping the question intact.  
- If the evidence contains a table name along with a column (e.g., `cards.name`), replace the table name and use pronouns or descriptive phrases to imply the source.
- If the evidence is empty, return an empty string.

**Input:**  
- **Database ID:** {db_id}  
- **Question:** {question}  
- **Evidence:** {evidence}  
- **SQL Query:** {sql_query}  
- **Candidate Table:** {candidate_table}  
- **Components:** {components}  

**Output Format:**  
New Question: [modified question]  
New Evidence: [modified evidence] (or empty string if none)
"""

    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]

    response = openai.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        max_tokens=300,
        temperature=0.7
    )

    generated_text = response.choices[0].message.content.strip()
    
    # Extract new question and evidence
    new_question, new_evidence = "", ""
    for row in generated_text.split("\n"):
        if row.startswith("New Question:"):
            new_question = row.replace("New Question: ", "").strip()
        elif row.startswith("New Evidence:"):
            new_evidence = row.replace("New Evidence: ", "").strip()

    return new_question, new_evidence

def generate_contextually_augmented_queries(db_id, subgraphs):
    """Processes queries by applying contextual augmentation and filtering unchanged ones."""
    
    rewritten_queries = []
    print(f"Generating contextually augmented questions for {db_id}...")

    for subgraph in subgraphs:
        for query in subgraph['equivalent_queries']:
            new_question, new_evidence = augment_with_contextual_synonyms(
                db_id, query['question'], query['evidence'], query['SQL'],
                subgraph['central_table'], subgraph['components']
            )

            # Ensure replacement occurred before keeping it
            if not any(comp in new_question.lower() for comp in subgraph['components']):
                query['new_question'], query['new_evidence'] = new_question, new_evidence
                rewritten_queries.append(query)

    return rewritten_queries

def hide_tables_with_contextual_augmentation(data):
    """Runs contextual augmentation on the dataset to conceal table names."""
    
    return {
        db_id: generate_contextually_augmented_queries(db_id, subgraphs)
        for db_id, subgraphs in data.items()
    }

# ---- Execution Block ----
train_queries_with_contextual_augmentation = hide_tables_with_contextual_augmentation(train_subgraphs_with_central_tables)
dev_queries_with_contextual_augmentation = hide_tables_with_contextual_augmentation(dev_subgraphs_with_central_tables)



Generating contextually augmented questions for codebase_community...
Generating contextually augmented questions for codebase_community...


### 4. Generating data for test set (THIS SHOULD BE DONE AT THE END)

In [82]:
# Now generating data for the test set
test_subgraphs_with_central_tables = process_dataset(test_subgraphs)
test_queries_with_synonym_replacement = hide_tables_with_synonym_replacement(test_subgraphs_with_central_tables)
test_queries_with_backtranslation = hide_tables_with_backtranslation(test_subgraphs_with_central_tables)
test_queries_with_contextual_augmentation = hide_tables_with_contextual_augmentation(test_subgraphs_with_central_tables)



Generating new questions for codebase_community...
Generating backtranslated questions for codebase_community...
Generating contextually augmented questions for codebase_community...


## 5. Final queries 


In [84]:
import os
import json

def save_questions_to_file(dict_queries, output_base_dir, technique):
    output_dir = f'{output_base_dir}/{technique}'
    os.makedirs(output_dir, exist_ok=True)  # Ensure the output directory exists
    original_questions = []
    new_questions = []
    
    # Iterate over db_ids and queries to collect original and new questions
    for db_id, queries in dict_queries.items():
        for i, query in enumerate(queries):
            original_entry = {
                "question_id": i,
                "db_id": db_id,
                "question": query['question'],
                "evidence": query['evidence'],
                "SQL": query['SQL'],
                "difficulty": query['difficulty']
            }
            new_entry = {
                "question_id": i,
                "db_id": db_id,
                "question": query['new_question'],
                "evidence": query['new_evidence'],
                "SQL": query['SQL'],
                "difficulty": query['difficulty']
            }
            original_questions.append(original_entry)
            new_questions.append(new_entry)

    # Save the questions to files
    with open(f'{output_dir}/original_questions.json', 'w') as f:
        json.dump(original_questions, f, indent=4)
    with open(f'{output_dir}/new_questions.json', 'w') as f:
        json.dump(new_questions, f, indent=4)
    
    print(f"Saved {len(original_questions)} original questions and {len(new_questions)} new questions for {db_id} to {output_dir}.")

def process_and_save_data_for_set(data_dict, output_base_dir, set_type):
    """Process and save questions for a specific dataset type (train, dev, test)."""
    for technique, data in data_dict.items():
        save_questions_to_file(data, output_base_dir, technique)

# Directory setup
output_base_dir = "../../../../data/rule_outputs/rule_3/"
os.makedirs(output_base_dir, exist_ok=True)  # Ensure the output directory exists

# Creating subdirectories for train, dev, and test
train_output_base_dir = f"{output_base_dir}/train"
dev_output_base_dir = f"{output_base_dir}/dev"
test_output_base_dir = f"{output_base_dir}/test"
os.makedirs(train_output_base_dir, exist_ok=True)
os.makedirs(dev_output_base_dir, exist_ok=True)
os.makedirs(test_output_base_dir, exist_ok=True)

# Data for each technique
train_data = {
    "synonym_replacement": train_queries_with_synonym_replacement,
    "backtranslation": train_queries_with_backtranslation,
    "contextual_augmentation": train_queries_with_contextual_augmentation
}

dev_data = {
    "synonym_replacement": dev_queries_with_synonym_replacement,
    "backtranslation": dev_queries_with_backtranslation,
    "contextual_augmentation": dev_queries_with_contextual_augmentation
}

test_data = {
    "synonym_replacement": test_queries_with_synonym_replacement,
    "backtranslation": test_queries_with_backtranslation,
    "contextual_augmentation": test_queries_with_contextual_augmentation
}

# Save the data for each dataset (train, dev, and test)
process_and_save_data_for_set(train_data, train_output_base_dir, "train")
process_and_save_data_for_set(dev_data, dev_output_base_dir, "dev")
process_and_save_data_for_set(test_data, test_output_base_dir, "test")


Saved 6 original questions and 6 new questions for codebase_community to ../../../../data/rule_outputs/rule_3//train/synonym_replacement.
Saved 6 original questions and 6 new questions for codebase_community to ../../../../data/rule_outputs/rule_3//train/backtranslation.
Saved 5 original questions and 5 new questions for codebase_community to ../../../../data/rule_outputs/rule_3//train/contextual_augmentation.
Saved 2 original questions and 2 new questions for codebase_community to ../../../../data/rule_outputs/rule_3//dev/synonym_replacement.
Saved 1 original questions and 1 new questions for codebase_community to ../../../../data/rule_outputs/rule_3//dev/backtranslation.
Saved 2 original questions and 2 new questions for codebase_community to ../../../../data/rule_outputs/rule_3//dev/contextual_augmentation.
Saved 1 original questions and 1 new questions for codebase_community to ../../../../data/rule_outputs/rule_3//test/synonym_replacement.
Saved 1 original questions and 1 new ques